# Classification

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib widget
plt.rcParams['figure.figsize'] = (12, 8)
from sklearn.metrics import accuracy_score
from sklearn.base import clone
from sklearn.multiclass import OneVsRestClassifier
from dataset import CovtypeDataset as Dataset
from eda.eda import EDA
from classification.pipeline import get_pipeline

# Logistic regression
from classification.logistic_regression.softmax import SoftmaxClassifier
from classification.logistic_regression.sigmoid import SigmoidClassifier
from classification.logistic_regression.irls import IRLSClassifier

# Discriminant Analysis
from classification.lda.lda import LDA
from classification.lda.qda import QDA
from classification.lda.fisher_ratio import fisher_ratio

# Advanced
from classification.advanced.probit import ProbitClassifier
from classification.advanced.laplace import LaplaceApprox

from evaluation.evaluator import Evaluator
from evaluation.visualizer import Visualizer

## Dataset

In [ ]:
d = Dataset()

### EDA

In [ ]:
eda = EDA(d)

#### Missing values

In [ ]:
print(eda.missing_values())

#### Target distribution

In [ ]:
print(eda.target_distribution())

In [ ]:
eda.plot_target_distribution()
plt.show()

#### Numeric features

In [ ]:
eda.plot_numeric_features()
plt.show()

#### Categorical features

In [ ]:
eda.plot_categorical_features()
plt.show()

#### Numeric features vs Target

In [ ]:
eda.plot_numeric_vs_target()
plt.show()

#### Preprocessing

In [ ]:
d.split()

## Models

### Logistic regression

#### Sigmoid

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# dataset
X, y = make_classification(
    n_samples=500_000,
    n_features=10,
    n_informative=6,
    n_classes=2,
    random_state=42
)

# split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# model
model = SigmoidClassifier(
    max_iter=20,
    step_size=4,
    batch_size=1024
)

# fit
model.fit(X_train, y_train, X_val=X_val, y_val=y_val)

# accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Acc = {acc:.4f}")

# ===== PLOT =====
epochs = range(len(model.train_loss_history_))

# 1. loss vs epoch
plt.figure(layout="constrained")
plt.plot(epochs, model.train_loss_history_, label="train")
plt.plot(epochs, model.val_loss_history_, label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.show()

# 2. loss vs time
plt.figure(layout="constrained")
plt.plot(model.time_history_, model.train_loss_history_, label="train")
plt.plot(model.time_history_, model.val_loss_history_, label="val")
plt.xlabel("Time (s)")
plt.ylabel("Loss")
plt.title("Loss vs Time")
plt.legend()
plt.show()

#### Newton-Raphson IRLS

In [ ]:
# model
model = IRLSClassifier()

# fit
model.fit(X_train, y_train, X_val=X_val, y_val=y_val)

# accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Acc = {acc:.4f}")

# ===== PLOT =====
epochs = range(len(model.train_loss_history_))

# 1. loss vs epoch
plt.figure(layout="constrained")
plt.plot(epochs, model.train_loss_history_, label="train")
plt.plot(epochs, model.val_loss_history_, label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.show()

# 2. loss vs time
plt.figure(layout="constrained")
plt.plot(model.time_history_, model.train_loss_history_, label="train")
plt.plot(model.time_history_, model.val_loss_history_, label="val")
plt.xlabel("Time (s)")
plt.ylabel("Loss")
plt.title("Loss vs Time")
plt.legend()
plt.show()

#### Softmax

In [ ]:
softmax = SoftmaxClassifier()
softmax.fit(d.X_train, d.y_train)
y_pred = softmax.predict(d.X_test)
y_true = d.y_test
acc = accuracy_score(y_true, y_pred)
print(f"Acc = {acc:.4f}")

### LDA & QDA

#### LDA

In [ ]:
model = get_pipeline(LDA(n_components=2))
model.fit(d.X_train, d.y_train)

In [ ]:
y_pred = model.predict(d.X_test)
y_true = d.y_test
acc = accuracy_score(y_true, y_pred)
print(f"Acc = {acc:.4f}")

#### Decision Boundary 2D

In [ ]:
model.named_steps["predictor"].plot2D(d.X_train, d.y_train, padding=2000)

#### QDA

In [ ]:
model = get_pipeline(QDA())
model.fit(d.X_train, d.y_train)

In [ ]:
y_pred = model.predict(d.X_test)
y_true = d.y_test
acc = accuracy_score(y_true, y_pred)
print(f"Acc = {acc:.4f}")

#### Fisher ratio

In [ ]:
J_scores = fisher_ratio(d.X_train, d.y_train)
df_ranking = pd.DataFrame({
    'Feature': d.feature_names,
    'Fisher_Ratio': J_scores
}).sort_values(by='Fisher_Ratio', ascending=False).reset_index(drop=True)
print(df_ranking)

### Perceptron & Regularization

## Advanced

### Probit

In [ ]:
from sklearn.datasets import make_blobs

centers = [[-4, 0], [0, 2]]
X, y = make_blobs(n_samples=1_000, centers=centers, cluster_std=2, random_state=40)
transformation = [[0.4, 0.3], [-0.4, 0.7]]
X = np.dot(X, transformation)

In [ ]:
def plot_decision_boundary_2features(model, X, y, f1=0, f2=1, resolution=200, title="Decision Boundary"):
    n, d = X.shape
    x_min, x_max = X[:, f1].min() - 1, X[:, f1].max() + 1
    y_min, y_max = X[:, f2].min() - 1, X[:, f2].max() + 1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))

    # Grid full dimension
    X_mean = np.mean(X, axis=0)
    grid = np.tile(X_mean, (xx.size, 1))
    grid[:, f1] = xx.ravel()
    grid[:, f2] = yy.ravel()

    Z = model.predict(grid).reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    plt.scatter(X[:, f1], X[:, f2], c=y, s=20, edgecolor="k", cmap="coolwarm")
    plt.xlabel(f"Feature {f1}")
    plt.ylabel(f"Feature {f2}")
    plt.title(title)
    plt.show()

def plot_probability_2features(model, X, y, f1=0, f2=1, class_idx=1, resolution=200, title="Probability"):
    n, d = X.shape
    x_min, x_max = X[:, f1].min() - 1, X[:, f1].max() + 1
    y_min, y_max = X[:, f2].min() - 1, X[:, f2].max() + 1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))

    X_mean = np.mean(X, axis=0)
    grid = np.tile(X_mean, (xx.size, 1))
    grid[:, f1] = xx.ravel()
    grid[:, f2] = yy.ravel()

    Z = model.predict_proba(grid)[:, class_idx].reshape(xx.shape)

    plt.contourf(xx, yy, Z, levels=50, cmap="viridis")
    plt.scatter(X[:, f1], X[:, f2], c=y, s=20, edgecolor="k", cmap="viridis")
    plt.colorbar(label=f"p(class={class_idx})")
    plt.xlabel(f"Feature {f1}")
    plt.ylabel(f"Feature {f2}")
    plt.title(title)
    plt.show()


sigmoid = SigmoidClassifier()
sigmoid.fit(X, y)

probit = ProbitClassifier()
probit.fit(X, y)

#### Decision boundary

In [ ]:
plot_decision_boundary_2features(sigmoid, X, y, f1=0, f2=1, title="Sigmoid Decision Boundary")
plt.show()

In [ ]:
plot_decision_boundary_2features(probit, X, y, f1=0, f2=1, title="Probit Decision Boundary")
plt.show()

#### Predict probability

In [ ]:
plot_probability_2features(sigmoid, X, y, f1=0, f2=1, class_idx=1, title="Softmax Probability Class 1")
plt.show()

In [ ]:
plot_probability_2features(probit, X, y, f1=0, f2=1, class_idx=1, title="Probit Probability Class 1")
plt.show()

#### Noise sensitivity

In [ ]:
def noisy_labels_test(model, X, y, noise_levels=[0.0, 0.1, 0.2, 0.3, 0.4], seed=42):
    np.random.seed(seed)

    accs = []
    classes = np.unique(y)

    for noise in noise_levels:
        y_noisy = y.copy()

        n_flip = int(len(y) * noise)
        flip_idx = np.random.choice(len(y), n_flip, replace=False)

        if len(classes) == 2:
            y_noisy[flip_idx] = 1 - y_noisy[flip_idx]
        else:
            y_noisy[flip_idx] = np.random.choice(classes, size=n_flip)

        m = clone(model)
        m.fit(X, y_noisy)

        y_pred = m.predict(X)

        accs.append(accuracy_score(y, y_pred))  # evaluate on clean labels

    print("noise level:", noise)
    print("changed labels:", np.mean(y_noisy == y))
    plt.plot(noise_levels, accs, marker="o")
    plt.xlabel("Fraction of label noise")
    plt.ylabel("Accuracy (clean labels)")
    plt.title(f"Robustness to label noise: {model.__class__.__name__}")
    plt.grid(True)
    plt.show()

    return accs

In [ ]:
# -------------------------------
# Độ nhạy với nhãn nhiễu
# -------------------------------
noisy_labels_test(SigmoidClassifier(), X, y)
noisy_labels_test(ProbitClassifier(), X, y)

## Laplace approximation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== Train Sigmoid (Logistic) =====
model = SigmoidClassifier()
list(model._fit(X, y))

# ===== Laplace Approximation =====
laplace = LaplaceApprox(model)
laplace.fit(X)

# ===== Plot decision boundary + uncertainty =====
fig, ax = plt.subplots(figsize=(6,6))
laplace.decision_boundary_sigma(X, y, ax=ax)
plt.show()

## Evaluation

In [ ]:
models = {
    "softmax": SoftmaxClassifier(),
    "LDA": LDA()
}
